# Projeto Airbnb Rio - Ferramenta de Previsão de Preço de Imóvel para pessoas comuns 

### Importar Bibliotecas e Bases de Dados

In [ ]:
import polars as pl
import polars.selectors as cs
import pathlib
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import train_test_split
import gc

In [ ]:
meses = {"jan":1, "fev":2, "mar":3, "abr":4, "mai":5, "jun":6, "jul":7, "ago":8, "set":9, "out":10, "nov":11, "dez":12}
caminho_bases = pathlib.Path("dataset")
pasta_parquet = pathlib.Path("dataset_parquet")
pasta_parquet.mkdir(exist_ok=True)

for arquivo in caminho_bases.iterdir():
    if arquivo.suffix == '.csv' and not arquivo.name.startswith('.'):
        nome_mes = arquivo.name[:3]
        mes = meses[nome_mes]
        ano = arquivo.name[-8:]
        ano = int(ano.replace(".csv", ""))
        df = pl.read_csv(arquivo, infer_schema_length=10000)
        df = df.with_columns(pl.lit(mes).alias("mes"))
        df = df.with_columns(pl.lit(ano).alias("ano"))
        nome_parquet = pasta_parquet / arquivo.with_suffix('.parquet').name
        df.write_parquet(nome_parquet)

In [ ]:
pasta_parquet = pathlib.Path("dataset_parquet")
base_airbnb_lazy = pl.scan_parquet(
    str(pasta_parquet / "*.parquet"), 
    extra_columns="ignore"
)

### Consolidar Base de Dados

In [ ]:
colunas= [
    'host_response_time', 'host_response_rate', 'host_is_superhost', 'host_listings_count', 
    'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 
    'bedrooms', 'beds', 'bed_type', 'amenities', 'price', 'security_deposit', 
    'cleaning_fee', 'guests_included', 'extra_people', 'minimum_nights', 'maximum_nights', 
    'number_of_reviews', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 
    'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 
    'review_scores_value', 'instant_bookable', 'is_business_travel_ready', 'cancellation_policy', 
    'ano', 'mes'
]

base_airbnb_filtrada = base_airbnb_lazy.select(colunas)

print(base_airbnb_filtrada.head(5).collect())

### Tratar Valores Faltando

In [ ]:
contagem_nulos = base_airbnb_filtrada.null_count().collect()
relatorio_nulos = contagem_nulos.transpose(include_header=True, header_name="Coluna", column_names=["Total de Nulos"])

colunas_com_muitos_nulos = relatorio_nulos.filter(
    pl.col("Total de Nulos") > 300000
)["Coluna"].to_list()

base_airbnb_limpa = base_airbnb_filtrada.drop(colunas_com_muitos_nulos)
base_airbnb_limpa = base_airbnb_limpa.drop("host_response_time", "host_response_rate")
base_airbnb_limpa = base_airbnb_limpa.drop_nulls()

print(base_airbnb_limpa.head(5).collect())

### Verificar Tipos de Dados em cada coluna

In [ ]:
print(base_airbnb_limpa.schema)

base_airbnb_tratada = base_airbnb_limpa.with_columns([
    
    pl.when(pl.col("price").str.contains(r"\.\d{2}$")) 
    .then(
        pl.col("price")
        .str.replace_all(r"[^\d]", "")                  
        .cast(pl.Float64) / 100.0          
    )
    .otherwise(
        pl.col("price")
        .str.replace_all(r"[^\d]", "")
        .cast(pl.Float64)
    ),
    
# Tratamento para 'extra_people'
    pl.when(pl.col("extra_people").str.contains(r"\.\d{2}$"))
    .then(
        pl.col("extra_people")
        .str.replace_all(r"[^\d]", "")
        .cast(pl.Float64) / 100.0
    )
    .otherwise(
        pl.col("extra_people")
        .str.replace_all(r"[^\d]", "")
        .cast(pl.Float64)
    )
])

print(base_airbnb_tratada.select(["price", "extra_people"]).head(5).collect())

### Análise Exploratória e Tratar Outliers

In [ ]:
colunas_numericas = [
    coluna for coluna, tipo in base_airbnb_tratada.schema.items() 
    if tipo.is_numeric()
]


df_numerico_pandas = base_airbnb_tratada.select(colunas_numericas).collect().to_pandas()

matriz_correlacao = df_numerico_pandas.corr()

plt.figure(figsize=(15, 10))
sns.heatmap(matriz_correlacao, annot=True, cmap="Greens", fmt=".2f", linewidths=0.5)

plt.show()

### Definição de Funções para Análise de Outliers

Vamos definir funções para auxiliar na análise de outliers das colunas.

In [ ]:
def limites(df, nome_coluna):
    valores_quartis = df.select([
        pl.col(nome_coluna).quantile(0.25).alias("q1"),
        pl.col(nome_coluna).quantile(0.75).alias("q3")
    ]).collect()
    
    q1 = valores_quartis["q1"][0]
    q3 = valores_quartis["q3"][0]
    
    amplitude = q3 - q1
    
    limite_inferior = q1 - 1.5 * amplitude
    limite_superior = q3 + 1.5 * amplitude
    
    return limite_inferior, limite_superior

def excluir_outliers(df, nome_coluna):
    qtde_linhas = df.collect().shape[0]
    lim_inf, lim_sup = limites(df, nome_coluna)
    df_filtrado = df.filter(
        (pl.col(nome_coluna) >= lim_inf) & (pl.col(nome_coluna) <= lim_sup)
    )
    linhas_removidas = qtde_linhas - df_filtrado.collect().shape[0]
    return df_filtrado, linhas_removidas

In [ ]:
def diagrama_caixa(df, nome_coluna):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    dados_coluna = df.select(nome_coluna).collect().get_column(nome_coluna)
    sns.boxplot(x=dados_coluna, ax=ax1)
    lim_inf, lim_sup = limites(df, nome_coluna)
    ax2.set_xlim([lim_inf, lim_sup])
    sns.boxplot(x=dados_coluna, ax=ax2)

def histograma(df, nome_coluna):
    plt.figure(figsize=(15, 5))
    dados_coluna = df.select(nome_coluna).collect().get_column(nome_coluna)
    sns.histplot(dados_coluna, kde=True)

def grafico_barra(df, nome_coluna):
    plt.figure(figsize=(15, 5))
    contagem_categorias = df.group_by(nome_coluna).len().collect()
    eixo_x = contagem_categorias.get_column(nome_coluna)
    eixo_y = contagem_categorias.get_column("len")
    ax = sns.barplot(x=eixo_x, y=eixo_y)
    lim_inf, lim_sup = limites(df, nome_coluna)
    ax.set_xlim([lim_inf, lim_sup])

Analisando a coluna "price"

In [ ]:
diagrama_caixa(base_airbnb_tratada, "price")
plt.show()

histograma(base_airbnb_tratada, "price")
plt.show()

In [ ]:
base_airbnb_tratada, linhas_deletadas = excluir_outliers(base_airbnb_tratada, "price")

print(f"Quantidade de linhas de outliers removidas: {linhas_deletadas}")

Analisando a coluna "extra_people"

In [ ]:
diagrama_caixa(base_airbnb_tratada, "extra_people")
plt.show()

histograma(base_airbnb_tratada, "extra_people")
plt.show()

In [ ]:
base_airbnb_tratada, linhas_deletadas = excluir_outliers(base_airbnb_tratada, "extra_people")

print(f"Quantidade de linhas de outliers removidas: {linhas_deletadas}")

Analisando a coluna "host_listings_count"

In [ ]:
diagrama_caixa(base_airbnb_tratada, "host_listings_count")
plt.show()

grafico_barra(base_airbnb_tratada, "host_listings_count")
plt.show()

In [ ]:
base_airbnb_tratada, linhas_deletadas = excluir_outliers(base_airbnb_tratada, "host_listings_count")

print(f"Quantidade de linhas de outliers removidas: {linhas_deletadas}")

Analisando a coluna "accommodates"

In [ ]:
diagrama_caixa(base_airbnb_tratada, "accommodates")
plt.show()

grafico_barra(base_airbnb_tratada, "accommodates")
plt.show()

In [ ]:
base_airbnb_tratada, linhas_deletadas = excluir_outliers(base_airbnb_tratada, "accommodates")

print(f"Quantidade de linhas de outliers removidas: {linhas_deletadas}")

Analisando a coluna "bathrooms"

In [ ]:
diagrama_caixa(base_airbnb_tratada, "bathrooms")
plt.show()

grafico_barra(base_airbnb_tratada, "bathrooms")
plt.show()

In [ ]:
base_airbnb_tratada, linhas_deletadas = excluir_outliers(base_airbnb_tratada, "bathrooms")

print(f"Quantidade de linhas de outliers removidas: {linhas_deletadas}")

Analisando a coluna "bedrooms"

In [ ]:
diagrama_caixa(base_airbnb_tratada, "bedrooms")
plt.show()

grafico_barra(base_airbnb_tratada, "bedrooms")
plt.show()

In [ ]:
base_airbnb_tratada, linhas_deletadas = excluir_outliers(base_airbnb_tratada, "bedrooms")

print(f"Quantidade de linhas de outliers removidas: {linhas_deletadas}")

Analisando a coluna "beds"

In [ ]:
diagrama_caixa(base_airbnb_tratada, "beds")
plt.show()

grafico_barra(base_airbnb_tratada, "beds")
plt.show()

In [ ]:
base_airbnb_tratada, linhas_deletadas = excluir_outliers(base_airbnb_tratada, "beds")

print(f"Quantidade de linhas de outliers removidas: {linhas_deletadas}")

Analisando a coluna "guests_included"

In [ ]:
diagrama_caixa(base_airbnb_tratada, "guests_included")
plt.show()

grafico_barra(base_airbnb_tratada, "guests_included")
plt.show()

In [ ]:
#Como essa feature aparenta representar um padrão de preenchimento do airbnb, resolvi 
#removê-la completamente da análise. A princípio, parece que os usuários não preenchem
#corretamente o campo de "guests included", mantendo o padrão de 1

base_airbnb_tratada = base_airbnb_tratada.drop("guests_included")

Analisando a coluna "minimum_nights"

In [ ]:
diagrama_caixa(base_airbnb_tratada, "minimum_nights")
plt.show()

grafico_barra(base_airbnb_tratada, "minimum_nights")
plt.show()

In [ ]:
base_airbnb_tratada, linhas_deletadas = excluir_outliers(base_airbnb_tratada, "minimum_nights")

print(f"Quantidade de linhas de outliers removidas: {linhas_deletadas}")

Analisando a coluna "maximum_nights"

In [ ]:
diagrama_caixa(base_airbnb_tratada, "maximum_nights")
plt.show()

grafico_barra(base_airbnb_tratada, "maximum_nights")
plt.show()

In [ ]:
#A feature "maximum nights" não parece ser boa para a nossa análise, portanto, vou removê-la

base_airbnb_tratada = base_airbnb_tratada.drop("maximum_nights")

Analisando a coluna "number_of_reviews"

In [ ]:
diagrama_caixa(base_airbnb_tratada, "number_of_reviews")
plt.show()

grafico_barra(base_airbnb_tratada, "number_of_reviews")
plt.show()

In [ ]:
#Como o objetivo da análise é auxiliar usuários novos na plataforma a calcularem preços,
#acredito que o número de reviews será pouco relevante, já que o novo usuário não teria
#nenhum review ainda. Portanto, vou remover essa feature da análise

base_airbnb_tratada = base_airbnb_tratada.drop("number_of_reviews")

### Tratamento de Colunas de Valores de Texto

Analisando a coluna "property_type"

In [ ]:
contagem_ordenada = (
    base_airbnb_tratada
    .group_by("property_type")
    .len()
    .sort("len", descending=True)
    .collect()
)

with pl.Config(tbl_rows=-1):
    print(contagem_ordenada)

In [ ]:
limite_minimo = 2000

base_airbnb_tratada = base_airbnb_tratada.with_columns([
    pl.when(
        pl.col("property_type").len().over("property_type") < limite_minimo
    )
    .then(pl.lit("Outros"))
    .otherwise(pl.col("property_type"))
    .alias("property_type")
])

In [ ]:
conferência_tipos = (
    base_airbnb_tratada
    .group_by("property_type")
    .len()
    .sort("len", descending=True)
    .collect()
)

with pl.Config(tbl_rows=-1):
    print(conferência_tipos)

Analisando a coluna "room_type"

In [ ]:
contagem_ordenada = (
    base_airbnb_tratada
    .group_by("room_type")
    .len()
    .sort("len", descending=True)
    .collect()
)

with pl.Config(tbl_rows=-1):
    print(contagem_ordenada)

Analisando a coluna "bed_type"

In [ ]:
contagem_ordenada = (
    base_airbnb_tratada
    .group_by("bed_type")
    .len()
    .sort("len", descending=True)
    .collect()
)

with pl.Config(tbl_rows=-1):
    print(contagem_ordenada)

In [ ]:
limite_minimo = 10000

base_airbnb_tratada = base_airbnb_tratada.with_columns([
    pl.when(
        pl.col("bed_type").len().over("bed_type") < limite_minimo
    )
    .then(pl.lit("Outros"))
    .otherwise(pl.col("bed_type"))
    .alias("bed_type")
])

In [ ]:
conferência_tipos = (
    base_airbnb_tratada
    .group_by("bed_type")
    .len()
    .sort("len", descending=True)
    .collect()
)

with pl.Config(tbl_rows=-1):
    print(conferência_tipos)

Analisando a coluna "cancellation_policy"

In [ ]:
contagem_ordenada = (
    base_airbnb_tratada
    .group_by("cancellation_policy")
    .len()
    .sort("len", descending=True)
    .collect()
)

with pl.Config(tbl_rows=-1):
    print(contagem_ordenada)

In [ ]:
limite_minimo = 10000

base_airbnb_tratada = base_airbnb_tratada.with_columns([
    pl.when(
        pl.col("cancellation_policy").len().over("cancellation_policy") < limite_minimo
    )
    .then(pl.lit("strict"))
    .otherwise(pl.col("cancellation_policy"))
    .alias("cancellation_policy")
])

In [ ]:
conferência_tipos = (
    base_airbnb_tratada
    .group_by("cancellation_policy")
    .len()
    .sort("len", descending=True)
    .collect()
)

with pl.Config(tbl_rows=-1):
    print(conferência_tipos)

Analisando a coluna "amenities"

In [ ]:
#Como temos um número elevado de amenities e, às vezes, a mesma amenity é escrita
#de maneiras diferentes, vou considerar a quantidade total de amenities em vez de
#considerar cada amenity separadamente

base_airbnb_tratada = base_airbnb_tratada.with_columns([
    pl.col("amenities").str.split(",").list.len().alias("n_amenities")
])

base_airbnb_tratada = base_airbnb_tratada.drop("amenities")

In [ ]:
#Como a coluna n_amenities é uma coluna numérica, agora vou analisá-la da mesma forma
#que fiz com as outras colunas numéricas e excluir os outliers, se for necessário

diagrama_caixa(base_airbnb_tratada, "n_amenities")
plt.show()

grafico_barra(base_airbnb_tratada, "n_amenities")
plt.show()

In [ ]:
base_airbnb_tratada, linhas_deletadas = excluir_outliers(base_airbnb_tratada, "n_amenities")

print(f"Quantidade de linhas de outliers removidas: {linhas_deletadas}")

### Visualização de Mapa das Propriedades

In [ ]:
amostra_df = base_airbnb_tratada.collect().sample(n=50000, seed=42)

centro_valores = amostra_df.select([
    pl.col("latitude").mean().alias("lat_media"),
    pl.col("longitude").mean().alias("lon_media")
])

centro_mapa = {
    "lat": centro_valores["lat_media"][0], 
    "lon": centro_valores["lon_media"][0]
}

amostra_pandas = amostra_df.to_pandas()

mapa = px.density_map(
    amostra_pandas, 
    lat='latitude', 
    lon='longitude', 
    z='price', 
    radius=2.5,
    center=centro_mapa, 
    zoom=10,
    map_style='open-street-map'
)

mapa.show()

### Encoding

É necessário ajustar as features para facilitar o trabalho do modelo que usaremos.

- Para as features de True/False, usaremos 0 para True e 1 para False
- Para as features de categorias (texto), usaremos o método de encoding de variáveis dummies

In [ ]:
colunas_tf = ["host_is_superhost", "instant_bookable", "is_business_travel_ready"]

base_airbnb_tratada = base_airbnb_tratada.with_columns([
    pl.when(pl.col(coluna) == "t").then(1)
    .when(pl.col(coluna) == "f").then(0)
    .otherwise(None)
    .alias(coluna)
    for coluna in colunas_tf
])

In [ ]:
colunas_categorias = ['property_type', 'room_type', 'bed_type', 'cancellation_policy']

base_df = base_airbnb_tratada.collect()

dummies_df = base_df.select(colunas_categorias).to_dummies(drop_first=False)

base_final_df = pl.concat(
    [base_df.drop(colunas_categorias), dummies_df], 
    how="horizontal"
)

base_airbnb_tratada = base_final_df.lazy()

### Modelo de Previsão

- Métricas de Avaliação

In [ ]:
def avaliar_modelo(nome_modelo, y_teste, previsao):
    r2 = r2_score(y_teste, previsao)
    rsme = np.sqrt(mean_squared_error(y_teste, previsao))
    return f"Modelo: {nome_modelo}\nR²: {r2}\nRSME: {rsme}"

- Escolha dos Modelos a Serem Testados
    1. Random Forest
    2. Linear Regression
    3. Extra Tree

In [ ]:
modelo_rf = RandomForestRegressor(n_jobs=2, random_state=10)
modelo_lr = LinearRegression()
modelo_et = ExtraTreesRegressor(n_jobs=2, random_state=10)

modelos = {"RandomForest": modelo_rf,
        "LinearRegression": modelo_lr,
        "ExtraTrees": modelo_et,
        }

y = base_airbnb_tratada.select("price")
x = base_airbnb_tratada.drop("price")

- Separação dos Dados em Treino e Teste + Treino do Modelo

In [ ]:
x_coletado = x.collect()
y_coletado = y.collect()

x_train, x_test, y_train, y_test = train_test_split(x_coletado, y_coletado, random_state=10)

del x_coletado
del y_coletado

gc.collect()

for nome_modelo, modelo in modelos.items():
    modelo.fit(x_train, y_train)
    previsao = modelo.predict(x_test)
    print(avaliar_modelo(nome_modelo, y_test, previsao))

### Análise do Melhor Modelo

O modelo escolhido foi o ExtraTreesRegression, pois apresentou maior valor de R² e menor valor de RSME, indicando boa taxa de acertos e erros não muito expressivos. O modelo RandomForestRegression também aprsentou bons resultados, mas um pouco inferiores ao ExtraTreesRegression. Já o modelo de regressão linear não se mostrou adequado para essa análise.

### Ajustes e Melhorias no Melhor Modelo

In [ ]:
importancia_features = pl.DataFrame({
    "Feature": x_train.columns,
    "Importancia": modelo_et.feature_importances_
})

importancia_features = importancia_features.sort("Importancia", descending=True)

plt.figure(figsize=(15, 6))

sns.barplot(
    x=importancia_features["Feature"].to_numpy(), 
    y=importancia_features["Importancia"].to_numpy(),
    hue=importancia_features["Feature"].to_numpy(),
    palette="viridis",
    legend=False
)

plt.xticks(rotation=90)
plt.title("Importância das Features - Modelo ExtraTrees")
plt.ylabel("Grau de Importância")
plt.tight_layout()

plt.show()

Agora, vou fazer ajustes no modelo com base nas informações que obtive acima

In [ ]:
#A feature "is_business_travel_ready" não parece ter impacto significativo no modelo escolhido,
#portanto, vou removê-la e testar novamento o modelo

base_teste = base_airbnb_tratada.drop("is_business_travel_ready")

y = base_teste.select("price")
x = base_teste.drop("price")

x_coletado = x.collect()
y_coletado = y.collect()

x_train, x_test, y_train, y_test = train_test_split(x_coletado, y_coletado, random_state=10)

del x_coletado
del y_coletado

gc.collect()


modelo_et.fit(x_train, y_train)
previsao = modelo_et.predict(x_test)

print(avaliar_modelo("ExtraTrees", y_test, previsao))

A remoção da coluna "is_business_travel_ready" se mostrou eficaz, já que praticamente não alterou as nossas métricas e tornou o modelo um pouco mais simples. Agora, vou testar com outras colunas

In [ ]:
base_teste = base_airbnb_tratada.drop(["is_business_travel_ready"])
base_teste = base_teste.drop(cs.contains("bed_type"))

y = base_teste.select("price")
x = base_teste.drop("price")

x_coletado = x.collect()
y_coletado = y.collect()

x_train, x_test, y_train, y_test = train_test_split(x_coletado, y_coletado, random_state=10)

del x_coletado, y_coletado
gc.collect()

modelo_et.fit(x_train, y_train)
previsao = modelo_et.predict(x_test)

print(avaliar_modelo("ExtraTrees - Sem Business e Sem Bed_Type", y_test, previsao))

Decidi remover essas colunas de tipo de cama pelos mesmos motivos citados anteriormente. Agora, vou consolidar a nossa base de dados.

In [ ]:
base_airbnb_final = base_teste.with_columns(pl.col("price"))

base_airbnb_final.collect().write_parquet("base_airbnb_final_completa.parquet")

### Deploy do Projeto

In [ ]:
import joblib

joblib.dump(modelo_et, "modelo.joblib")